## Structured Output
Models can be requested to provide their response in a format matching a given schema.

In [8]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001BD288B7590>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001BD288B7950>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="This year the movie was released")
    director:str = Field(description="The director of the movie")
    rating: float = Field(description="The movie rating out of 10")

In [10]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001BD288B7590>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001BD288B7950>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'pa

In [11]:
model.invoke("Provide the details about movie Inception")

AIMessage(content='**Inception (2010) – Quick Reference**\n\n| Item | Details |\n|------|---------|\n| **Title** | *Inception* |\n| **Release Date** | July\u202f16,\u202f2010 (United States) |\n| **Running Time** | 148\u202fminutes |\n| **Genre** | Science‑fiction, Action, Thriller |\n| **Director / Writer** | Christopher Nolan |\n| **Producer(s)** | Emma Thomas, Christopher Nolan, Charles Roven, Jordan Goldberg |\n| **Production Companies** | Warner Bros. Pictures, Legendary Pictures, Syncopy Inc. |\n| **Distribution** | Warner Bros. Pictures |\n| **Budget** | ≈\u202f$160\u202fmillion (USD) |\n| **Box‑Office Gross** | ≈\u202f$836\u202fmillion worldwide (≈\u202f$292\u202fmillion domestic, $544\u202fmillion international) |\n| **MPAA Rating** | PG‑13 (Violence, language, some brief drug use) |\n| **Format** | 2‑D, 3‑D (IMAX), Digital, 4K Blu‑ray, DVD, streaming (various platforms) |\n\n---\n\n## Synopsis\n\n*Inception* follows **Dom Cobb** (Leonardo DiCaprio), a highly skilled “extracto

In [12]:
model_with_structure.invoke("Provide the details about movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [14]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="This year the movie was released")
    director:str = Field(description="The director of the movie")
    rating: float = Field(description="The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)

response = model_with_structure.invoke("Provide the details about movie Inception")
response


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide the details about movie Inception". We have a function definition for Movie that takes director, rating, title, year. We should call it with appropriate data. Inception details: director Christopher Nolan, rating maybe 8.8 (IMDb), year 2010. Provide details. Use function.', 'tool_calls': [{'id': 'fc_2d835e59-0721-467b-b576-362ce9180209', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 158, 'total_tokens': 277, 'completion_time': 0.253434074, 'completion_tokens_details': {'reasoning_tokens': 67}, 'prompt_time': 0.044584772, 'prompt_tokens_details': None, 'queue_time': 0.291779795, 'total_time': 0.298018846}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_87b3c396db', 'service_tier': 'on_demand', 'finish

### Nested Structure

In [15]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in million USD")

In [20]:
model_with_structure  = model.with_structured_output(MovieDetails)


response = model_with_structure.invoke("Provide the details about movie Inception")
response



MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Pete Postlethwaite', role='Maurice Fischer')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160000000.0)

### TypedDict
TypeDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [21]:
from typing_extensions import  TypedDict, Annotated


In [24]:
class MovieDict(TypedDict):
    """A Movie with details"""
    title:str = Annotated[str, ..., "The title of the movie"]
    year:int = Annotated[int, ..., "This year the movie was released"]
    director:str = Annotated[str, ..., "The director of the movie"]
    rating: float = Annotated[float, ..., "movie rating out of 10"]


In [26]:
Model_withtypedict = model.with_structured_output(MovieDict)
response = Model_withtypedict.invoke("Please provide the details of the avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [27]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClass
A data class is a class typically containing mainly data, although there aren't really any restriction. You create it using the @dataclass decorator.

In [8]:
import os
from langchain.chat_models import init_chat_model

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

model = init_chat_model("groq:openai/gpt-oss-120b")


In [19]:
# pydantic
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Person's Contact Information"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email Address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model  ,
    response_format=ContactInfo 

)

result = agent.invoke({
    "messages": [{"role":"user", "content":"Extract contactinfo from: John Doe, john@example.com, (555) 123- 4567"}]
})
result

{'messages': [HumanMessage(content='Extract contactinfo from: John Doe, john@example.com, (555) 123- 4567', additional_kwargs={}, response_metadata={}, id='e4853c33-2b35-44cd-bab9-9e411837c5df'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123- 4567"}', additional_kwargs={'reasoning_content': 'The user wants extraction of contact info. We need to output JSON matching schema ContactInfo. Must include name, email, phone. Ensure compact JSON, all required fields. Provide "John Doe", "john@example.com", "(555) 123- 4567". Probably trim spaces? Keep as given. Provide JSON.\n\n'}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 238, 'total_tokens': 341, 'completion_time': 0.217406443, 'completion_tokens_details': {'reasoning_tokens': 66}, 'prompt_time': 0.052415004, 'prompt_tokens_details': None, 'queue_time': 0.310395804, 'total_time': 0.269821447}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_77b12279f9',

In [20]:
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123- 4567')

In [22]:
#typedict

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Person's Contact Information"""
    name: str # The name of the person"
    email: str # The email Address of the person"
    phone: str # The phone number of the person"

agent = create_agent(
    model  ,
    response_format=ContactInfo 

)

result = agent.invoke({
    "messages": [{"role":"user", "content":"Extract contactinfo from: John Doe, john@example.com, (555) 123- 4567"}]
})
result['structured_response']

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [24]:
# dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Person's Contact Information"""
    name: str # The name of the person"
    email: str # The email Address of the person"
    phone: str # The phone number of the person"

agent = create_agent(
    model  ,
    response_format=ContactInfo 

)

result = agent.invoke({
    "messages": [{"role":"user", "content":"Extract contactinfo from: John Doe, john@example.com, (555) 123- 4567"}]
})
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123- 4567')